# Create OCAST Awards (Oklahoma Center for the Advancement of Science and Technology)

Creates OCAST awards from the award lists OCAST publishes as PDF tables
(parsed by word coordinates in `scripts/local/ocast_to_s3.py`; oklahoma.gov
robots.txt allows all agents):

- `{2014..2018}awards.pdf` (oklahoma.gov/content/dam/ok/en/ocast/documents/about/projects/):
  project number, PI, organisation, title, award amount (USD)
- `2013awards.pdf` (same format; only on the retired ok.gov site, read from
  the Internet Archive capture of 2018-07-07)
- `2018-2021-projects.pdf` (FY2018-FY2021, all programmes): project number,
  PI first/last name, organisation, title, project start date, partner organisation
- `Website Awards2024.pdf`: Industry Innovation Program 2024 (PI, company,
  title, total budget; no project numbers)

**522 awards, FY2013-FY2021 + 2024.** Programmes from the project-number
prefix (as the lists' captions name them): HR Health Research (214), AR
Oklahoma Applied Research Support / OARS (124), IP/AP Intern Partnerships
(108, training: student interns placed in Oklahoma companies, in scope per the
Mitacs precedent), PS Plant Science Research (49), HF Health Research
Fellowship (7), Industry Innovation Program (20, company R&D grants, flagged).
No filter: every listed award is kept.

**Not covered:** OCAST awards before FY2013 (not published as lists online;
2,300+ projects since 1987) and FY2022-2023 (no list published). Amounts are
missing for FY2019-FY2021 (the multi-year list has no amount column), so
pct_amount is ~78%.

**`funder_award_id` (§2.1.1):** the OCAST project number (`HR14-012`,
`AR131-033`, `IP19.2-007`), the form citing works acknowledge (`HR##-###` is
the dominant stub shape); 410 existing stub rows (150 ids) collapse onto these
awards. The 2024 Industry Innovation awards have no published number and ship
a synthetic `OCAST-II-2024-{md5(org|title)[:10]}`.

**Dates:** `start_date` from the multi-year list (FY2018-2021, 33%);
`start_year` = the fiscal year in the project number (FY = 20yy).

**Prerequisites:** run `scripts/local/ocast_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/ocast/ocast_projects.parquet` (§1.4 shrink guard).

**Funder details (F4320* path):**
- funder_id: `4320310627`
- display_name: Oklahoma Center for the Advancement of Science and Technology
- ror_id / doi: from `openalex.funders.funders` (ror 02q578v20, doi 10.13039/100008569)

**Priority:** `593` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ocast_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ocast/ocast_projects.parquet`;

In [ ]:
%sql
SELECT program, fiscal_year, COUNT(*) as n
FROM openalex.awards.ocast_raw
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
DESCRIBE openalex.awards.ocast_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ocast_raw LIMIT 5;

## Step 1.6: Funder existence check (F4320* path)
F4320310627 must resolve to exactly 1 row in `openalex.funders.funders`.
If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320310627;

## Step 2: Create OCAST Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ocast_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320310627  -- OCAST
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
    CASE WHEN NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    g.program as funder_scheme,
    'ocast_award_lists' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    COALESCE(YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')), TRY_CAST(g.fiscal_year AS INT)) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', TRIM(g.lead_family_name),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.organization), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>)
        as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>)
        as investigators,
    CASE g.source_pdf
        WHEN '2013awards.pdf' THEN 'https://web.archive.org/web/20180707215340/https://www.ok.gov/ocast/documents/2013awards.pdf'
        WHEN 'Website%20Awards2024.pdf' THEN 'https://oklahoma.gov/content/dam/ok/en/ocast/documents/Website%20Awards2024.pdf'
        ELSE CONCAT('https://oklahoma.gov/content/dam/ok/en/ocast/documents/about/projects/', g.source_pdf)
    END as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.ocast_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): OCAST project numbers (HR14-012, AR131-033, IP19.2-007)
-- or the synthetic 2024 Industry Innovation key. Expect 0 rows.
SELECT funder_award_id, display_name
FROM openalex.awards.ocast_awards
WHERE NOT (funder_award_id RLIKE '^[A-Z]{2}[0-9]{2,3}([.][0-9])?-[0-9]{2,3}$'
           OR funder_award_id RLIKE '^OCAST-II-2024-[0-9a-f]{10}$');

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ocast_award_lists' AND priority = 593;

-- Insert into openalex_awards_raw with priority.
-- Priority 593: direct-from-funder ingest of OCAST's published award lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 593 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    593 as priority
FROM openalex.awards.ocast_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_ocast_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.ocast_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, landing_page_url
FROM openalex.awards.ocast_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.ocast_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(amount) as with_amount, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.ocast_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check. Intern Partnership supervisors (e.g. University
-- of Tulsa engineering faculty) legitimately hold many small intern awards.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       COUNT(*) AS n, collect_set(funder_scheme) AS schemes
FROM openalex.awards.ocast_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.ocast_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect 100% title/PI, ~78% amount, 33% start_date).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.ocast_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement stubs (priority 0) that these awards upgrade.
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(b.id) as upgraded_by_ocast_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.ocast_awards b ON b.id = c.id
WHERE c.funder_id = 4320310627 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ocast_award_lists'
GROUP BY provenance, priority;